In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.cluster import KMeans

In [ ]:
df=pd.read_csv('/content/StudentsPerformance.csv')

In [ ]:
score_df = df.copy()
score_df.head()

In [ ]:
score_df.rename(columns={"race/ethnicity":"ethnicity","parental level of education":"parent_education"
                    ,"math score":"math","reading score":"reading","writing score":"writing",
                    "test preparation course":"pre"},inplace=True)
score_df.head()

In [ ]:
score_df.dtypes

In [ ]:
fig, ax = plt.subplots()
fig.subplots_adjust(hspace=0.8, wspace=0.8, left = 0.2, right = 1.5)
for idx in range(3):
    plt.subplot(1,3, idx+1)
    ethn_df = score_df.groupby("ethnicity")[list(score_df.columns[-3:])[idx]].mean()
    sns.barplot(x=ethn_df.index, y = ethn_df.values, palette = "Greens")
    plt.xlabel("Group")
    plt.ylabel("mean score")
    plt.xticks(rotation=90)
    plt.title(list(score_df.columns[-3:])[idx])
plt.show()

In [ ]:
labelencoder = LabelEncoder()
train_df = score_df.copy()
train_df["parent_education"] = labelencoder.fit_transform(train_df["parent_education"])
train_df["pre"] = labelencoder.fit_transform(train_df["pre"])
train_df["lunch"] = labelencoder.fit_transform(train_df["lunch"])
train_df.head()

In [ ]:
kmeans_dis = list()
for idx in range(2, min(25, len(train_df) + 1)):
    kmeans = KMeans(init='k-means++', n_clusters=idx, n_init=20, random_state=42)
    kmeans.fit(train_df.iloc[:, 2:])
    kmeans_dis.append(kmeans.inertia_)
plt.plot(list(range(2, min(25, len(train_df) + 1))), kmeans_dis, marker='o')
plt.xlabel('Number of clusters')
plt.ylabel('Summation of distance')
plt.show()

In [ ]:
kmeans = KMeans(init='k-means++', n_clusters=8, n_init=20, random_state=42)
kmeans.fit(train_df.iloc[:, 2:])
kmeans_label = kmeans.labels_
score_df['classification'] = kmeans_label
score_df.head(10)

In [ ]:
class_df = score_df.groupby("classification")[score_df.columns[-4:-1]].mean()
class_df

In [ ]:
ind = np.arange(8)
width = 0.35
fig, ax = plt.subplots()
rects1 = ax.bar(ind - width/2, class_df.math, width, label='Math')
rects2 = ax.bar(ind, class_df.reading, width, label='Reading')
rects3 = ax.bar(ind + width/2, class_df.writing, width, label='Writing')

ax.set_xlabel('Classiffication')
ax.set_ylabel('Scores')
ax.set_xticks(ind)
ax.legend()
plt.show()

In [ ]:
class_df["total_ave_score"] = (class_df.math + class_df.reading + class_df.writing)/3
rank = class_df["total_ave_score"].sort_values(ascending = False)
rank.index

In [ ]:
rank

In [ ]:
def plot_pie_chart(column):
    fig, ax = plt.subplots(figsize=(20,16))
    color = ['orange','lightblue','green','yellow','red','pink','brown','gray']
    for idx, cluster in enumerate(rank.index):
        plt.subplot(3, 3, idx+1)
        counts = score_df.loc[score_df['classification'] == cluster, column].value_counts()
        plt.pie(counts.values, labels=counts.index, autopct='%1.1f%%', startangle=90, colors=color[:len(counts)])
        plt.axis('equal')
        plt.title('Rank ' + str(idx))
    plt.show()
plot_pie_chart('parent_education')

# New Section

In [ ]:
def plot_bar_chart(column):
    counts = pd.crosstab(score_df['classification'], score_df[column]).reindex(rank.index, fill_value=0)
    counts.index = range(len(counts))
    ax = counts.plot.bar(figsize=(8,6))
    ax.set_xlabel('Rank')
    ax.set_ylabel('# of students')
    ax.set_title('Counts of ' + column)
    plt.tight_layout()
    plt.show()
plot_bar_chart('pre')

In [ ]:
plot_bar_chart("gender")